# Dyna-Q, Dyna-Q+ et Deep Dyna : planifier avec un modèle du monde

| Propriété        | Valeur                                           |
|------------------|--------------------------------------------------|
| Famille          | Model-based RL                                   |
| Environnements   | CartPole-v1, Gridworld non-stationnaire          |
| Espace d'actions | Discret                                          |
| Précédents       | Q-learning (notebook 01), DQN, SAC               |

---

## 1. Problème et motivation

### Le coût de l'expérience réelle

Dans les méthodes model-free comme DQN ou SAC, chaque transition $(s, a, r, s')$
est **utilisée une seule fois**, puis jetée ou stockée dans un replay buffer.

Ce design suppose que les transitions sont bon marché. C'est **faux** dans la réalité :

- un robot physique s'use et prend du temps à exécuter une trajectoire ;
- une expérience de laboratoire coûte de l'énergie, du matériel ou des patients ;
- un système de recommandation ne peut pas tester des millions d'actions sur de vrais utilisateurs.

### L'idée model-based

Et si on pouvait **apprendre un modèle du monde** — une fonction $\hat{T}(s, a) = (r, s')$
— et s'en servir pour *simuler* des transitions supplémentaires sans jamais toucher
l'environnement réel ?

C'est exactement le principe de **Dyna** (Sutton, 1990) :

> Entre deux pas réels, l'agent *planifie* en tirant des transitions depuis son
> modèle appris, et les utilise pour mettre à jour sa Q-table exactement comme s'il
> les avait vécues.

### Situer Dyna dans la progression

```
Model-free                          Model-based
────────────────────────────────────────────────────────
Q-learning (01) → DQN → SAC   →   Dyna-Q → PILCO → PETS → Dreamer
                                    ↑
```

Dyna est la brique d'entrée du RL model-based : simple à comprendre, facile à
implémenter, et suffisant pour montrer le gain en **efficacité-échantillon**.


## 2. Intuition

### Carte mentale Dyna

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    classDef primary fill:none,stroke:#2563eb,stroke-width:2px
    classDef secondary fill:none,stroke:#d97706,stroke-width:2px
    classDef result fill:none,stroke:#16a34a,stroke-width:2px

    env["Environnement reel"]:::primary

    subgraph real["Apprentissage depuis l'experience reelle"]
        q["Q-table / Q-network"]:::primary
        td_real["Update TD direct"]:::result
    end

    subgraph planning["Planning depuis le modele appris"]
        model["Modele du monde"]:::secondary
        imagined["Transitions simulees"]:::secondary
    end

    update["Meme regle TD"]:::result
    agent["Agent ameliore"]:::result

    env -->|"transition $$ (s,a,r,s') $$"| q
    env -->|"transition $$ (s,a,r,s') $$"| model
    q --> td_real
    td_real --> update
    model --> imagined
    imagined --> update
    update --> agent
```


### Analogie

Imaginez un étudiant qui prépare un examen.

- **Expérience réelle** : faire de vrais exercices, corriger ses erreurs.
- **Planification (révision mentale)** : entre deux séances, rejouer mentalement
  les exercices déjà vus.

Dyna fait exactement ça : quelques vraies expériences, puis beaucoup de révisions
mentales à partir du modèle mémorisé.

### Rôle de chaque brique

| Brique            | Rôle                                                      |
|-------------------|-----------------------------------------------------------|
| **Q-table**       | Mémorise la valeur $Q(s, a)$ pour chaque état-action      |
| **Modèle M**      | Mémorise $(r, s')$ pour chaque $(s, a)$ déjà vu           |
| **Update réelle** | TD(0) sur la vraie transition — corrige avec les faits    |
| **Planning**      | TD(0) sur des transitions tirées de M — amplifie le signal |
| **$k$ pas**       | Nombre de mises à jour simulées par pas réel (levier clé) |

Plus $k$ est grand, plus on exploite le modèle — au risque de propager ses erreurs.


## 3. Environnement : CartPole-v1

CartPole simule un pendule inversé sur un chariot.

| Propriété          | Valeur                                        |
|--------------------|-----------------------------------------------|
| Observation $s$    | $[x,\, \dot{x},\, \theta,\, \dot{\theta}]$ — 4 flottants |
| Actions            | $\{0 = \text{gauche},\; 1 = \text{droite}\}$  |
| Reward             | $+1$ à chaque pas de temps                    |
| Fin d'épisode      | $|\theta| > 12°$, $|x| > 2.4$, ou $t = 500$  |
| Score optimal      | 500 (pas sans tomber)                         |

C'est un environnement discret en actions mais **continu en observations**, ce qui
nous obligera à **discrétiser** l'espace d'état pour le cas tabulaire.


In [ ]:
import math
import random
from pathlib import Path
from collections import defaultdict, deque
from dataclasses import dataclass
from tqdm.auto import tqdm
from IPython.display import Video, display

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


def moving_average(values, window=10):
    if len(values) == 0:
        return np.array([])
    w = max(1, min(window, len(values)))
    kernel = np.ones(w) / w
    if len(values) < w:
        return np.asarray(values, dtype=float)
    head = np.asarray(values[: w - 1], dtype=float)
    tail = np.convolve(np.asarray(values, dtype=float), kernel, mode="valid")
    return np.concatenate([head, tail])


def plot_learning_curve(ax, values, title, ylabel, color, ma_window=10):
    x = np.arange(1, len(values) + 1)
    ax.plot(x, values, color=color, alpha=0.30, linewidth=1.2)
    ax.plot(x, moving_average(values, ma_window), color=color, linewidth=2.2)
    ax.set_title(title)
    ax.set_xlabel("Episode")
    ax.set_ylabel(ylabel)


def epsilon_greedy(values, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(len(values)))
    best_value = np.max(values)
    best_actions = np.flatnonzero(np.isclose(values, best_value))
    return int(rng.choice(best_actions))

In [ ]:
# Episode aleatoire court pour voir ce que l'agent recoit
env_demo = gym.make('CartPole-v1')
obs, _ = env_demo.reset(seed=0)
print('Observation initiale :', obs)
print('Forme obs            :', obs.shape)
print('Espace actions       :', env_demo.action_space)
print()
print('3 premieres transitions aleatoires :')
for step in range(3):
    action = env_demo.action_space.sample()
    next_obs, reward, terminated, truncated, info = env_demo.step(action)
    print(f'  step {step+1} | action={action} | reward={reward:.1f} | '
          f'x={next_obs[0]:.3f} theta={next_obs[2]:.4f} | done={terminated or truncated}')
    obs = next_obs
env_demo.close()


### Discrétisation de CartPole pour le cas tabulaire

Une Q-table est indexée par des **entiers**. L'observation continue de CartPole
doit donc être convertie en un tuple d'indices discrets.

| Dimension        | Borne basse | Borne haute | Nb de bins |
|------------------|-------------|-------------|------------|
| $x$              | −2.4        | 2.4         | 8          |
| $\dot{x}$        | −3.0        | 3.0         | 10         |
| $\theta$         | −0.2095     | 0.2095      | 16         |
| $\dot{\theta}$   | −3.5        | 3.5         | 16         |

Taille totale : $8 \times 10 \times 16 \times 16 \times 2 = 40{,}960$ entrées.
Les observations hors bornes sont clippées.


In [ ]:
class CartPoleDiscretizer:
    """Transforme une observation continue de CartPole en tuple d'indices entiers."""

    def __init__(self, bins=(8, 10, 12, 12)):
        # Bornes pédagogiques alignées avec le package : assez larges pour éviter de saturer
        # trop vite les vitesses, mais assez compactes pour que la table soit visitable.
        lows  = np.array([-2.4, -2.5, -0.2095, -4.0], dtype=np.float32)
        highs = np.array([ 2.4,  2.5,  0.2095,  4.0], dtype=np.float32)
        self.edges = [
            np.linspace(low, high, count, dtype=np.float32)
            for low, high, count in zip(lows, highs, bins)
        ]

    @property
    def shape(self):
        """Forme de la Q-table sans la dimension action."""
        return tuple(len(edge) for edge in self.edges)

    def transform(self, obs):
        """Observation -> tuple d'entiers, avec clipping aux bornes de discrétisation."""
        obs = np.asarray(obs, dtype=np.float32)
        indices = []
        for value, edge in zip(obs, self.edges):
            idx = int(np.digitize(value, edge) - 1)
            idx = int(np.clip(idx, 0, len(edge) - 1))
            indices.append(idx)
        return tuple(indices)

In [ ]:
# Mini-test CartPoleDiscretizer
disc = CartPoleDiscretizer()
print('Forme Q-table (sans action) :', disc.shape)
s = disc.transform(np.array([0.1, 0.5, 0.05, -0.3]))
print('Obs [0.1, 0.5, 0.05, -0.3]  ->', s)
assert len(s) == 4 and all(isinstance(i, int) for i in s)
s_clip = disc.transform(np.array([99.0, 99.0, 99.0, 99.0]))
print('Obs hors bornes [99, ...]    ->', s_clip, '(clippe aux max)')
assert s_clip == tuple(len(e) - 1 for e in disc.edges)
print('CartPoleDiscretizer mini-test OK.')


## 4. Équations

### 4.1 Mise à jour Dyna-Q (TD(0))

La Q-table est mise à jour selon la règle TD(0), identique à Q-learning :

$$
Q(s, a) \;\leftarrow\; Q(s, a) \;+\; \alpha \,\Bigl[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\Bigr]
$$

| Terme                              | Rôle                                                       |
|------------------------------------|------------------------------------------------------------|
| $\alpha \in (0, 1]$                | Taux d'apprentissage                                       |
| $\gamma \in [0, 1)$                | Facteur de discount                                        |
| $r$                                | Récompense observée (réelle ou simulée)                    |
| $\max_{a'} Q(s', a')$              | Bootstrap : meilleure valeur estimée de l'état suivant     |
| $r + \gamma \max_{a'} Q(s', a')$   | Cible TD — l'estimation corrigée de $Q(s, a)$             |
| $\delta = \text{cible} - Q(s, a)$  | Erreur TD — signal d'apprentissage                        |

**Point clé de Dyna** : cette mise à jour est appliquée identiquement aux
transitions **réelles** et **simulées** (tirées du modèle appris).


### 4.2 Bonus d'exploration Dyna-Q+

Si l'environnement **change**, les vieilles transitions du modèle deviennent
stériles. Dyna-Q+ corrige cela par un **bonus d'exploration lors du planning** :

$$
r^+ = r + \kappa \sqrt{\tau}
$$

| Terme        | Rôle                                                                 |
|--------------|----------------------------------------------------------------------|
| $r$          | Récompense mémorisée pour $(s, a)$                                   |
| $\kappa > 0$ | Poids du bonus — petite constante ($10^{-3}$ à $10^{-2}$)           |
| $\tau$       | Nombre de pas depuis la **dernière visite réelle** de $(s, a)$       |
| $\sqrt{\tau}$ | Croissance lente — même un grand $\tau$ ne domine pas $r$           |

**Intuition** : si $(s, a)$ n'a pas été visité depuis longtemps, le modèle
associé est peut-être obsolète. Le bonus rend cet état-action *attractif* lors
du planning, forçant l'agent à le retester en vrai.


### 4.3 Deep Dyna : pertes combinées

Quand l'espace d'état est continu, on remplace la Q-table par $Q_\theta$ et
le modèle tabulaire par un réseau de dynamiques $M_\phi$.

**Perte Bellman** (comme DQN) :

$$
\mathcal{L}_Q(\theta) = \mathbb{E}\bigl[(r + \gamma \max_{a'} Q_{\bar\theta}(s', a') - Q_\theta(s, a))^2\bigr]
$$

**Perte du modèle** :

$$
\mathcal{L}_M(\phi) = \|\hat{s}' - s'\|^2 \;+\; \|\hat{r} - r\|^2 \;+\; \text{BCE}(\hat{d}, d)
$$

| Terme      | Nature         | Explication                           |
|------------|----------------|---------------------------------------|
| MSE état   | Régression     | Prédire le prochain état (continu)    |
| MSE reward | Régression     | Prédire la récompense scalaire        |
| BCE done   | Classification | Prédire la fin d'épisode (0/1)        |

La prédiction résiduelle $\hat{s}' = s + \Delta_\phi(s, a)$ est plus facile à
apprendre car les variations entre deux pas sont petites pour CartPole.


## 5. Composants from-scratch


### Brique 1 : QTable

Tableau NumPy de shape `(*state_shape, action_count)`.
La méthode `update` applique : $Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \delta$.


In [ ]:
class QTable:
    """Table Q indexee par un tuple d entiers et une action entiere."""

    def __init__(self, state_shape, action_count):
        self.state_shape = tuple(state_shape)
        self.action_count = int(action_count)
        self.values = np.zeros((*self.state_shape, self.action_count), dtype=np.float32)

    def get(self, state):
        """Renvoie Q(s, .) pour toutes les actions."""
        return self.values[state]

    def update(self, state, action, target, alpha):
        """Q(s,a) <- Q(s,a) + alpha*(target - Q(s,a)). Renvoie delta."""
        old_value = float(self.values[state + (action,)])
        td_error = float(target) - old_value
        self.values[state + (action,)] = old_value + alpha * td_error
        return td_error


In [ ]:
# Mini-test QTable
q_table = QTable(state_shape=(3, 4), action_count=2)
print('Forme values   :', q_table.values.shape)
assert q_table.values.shape == (3, 4, 2)
q_table.values[1, 2] = np.array([0.5, -0.3], dtype=np.float32)
delta = q_table.update((1, 2), 0, target=1.0, alpha=0.1)
print(f'Ancienne Q(1,2,0)=0.5 | cible=1.0 | delta={delta:.4f}')
expected = 0.5 + 0.1 * (1.0 - 0.5)
assert abs(q_table.values[1, 2, 0] - expected) < 1e-5
print('QTable mini-test OK.')


### Brique 2 : TabularWorldModel

**Dictionnaire** $(s, a) \to (r, s', \text{done}, \text{last\_seen})$.
Modèle **déterministe** et **à un pas** : ne gère pas l'incertitude.


In [ ]:
@dataclass
class ModelTransition:
    state: tuple
    action: int
    reward: float
    next_state: tuple
    done: bool
    last_seen: int


class TabularWorldModel:
    """Dictionnaire (s, a) -> (r, s_prime, done, last_seen). Modele deterministe."""

    def __init__(self):
        self.transitions = {}

    def update(self, state, action, reward, next_state, done, time_step):
        key = (tuple(state), int(action))
        self.transitions[key] = ModelTransition(
            state=tuple(state), action=int(action), reward=float(reward),
            next_state=tuple(next_state), done=bool(done), last_seen=int(time_step),
        )

    def ensure_all_actions_for_state(self, state, action_count, time_step):
        """Pour Dyna-Q+ : initialise les actions non essayees avec r=0, s_prime=s."""
        state = tuple(state)
        for action in range(action_count):
            key = (state, action)
            if key not in self.transitions:
                self.transitions[key] = ModelTransition(
                    state=state, action=action, reward=0.0,
                    next_state=state, done=False, last_seen=int(time_step),
                )

    def sample(self, rng):
        """Tire une transition uniformement au hasard. Renvoie None si vide."""
        if not self.transitions:
            return None
        keys = list(self.transitions.keys())
        return self.transitions[keys[int(rng.integers(len(keys)))]]

    def __len__(self):
        return len(self.transitions)


In [ ]:
# Mini-test TabularWorldModel
model = TabularWorldModel()
model.update((0, 1), 1, 1.0, (0, 2), False, time_step=3)
model.ensure_all_actions_for_state((0, 1), action_count=3, time_step=3)
assert len(model) == 3
t = model.sample(np.random.default_rng(0))
assert t is not None and t.state == (0, 1) and t.action in {0, 1, 2}
print(f'Taille modele : {len(model)} (3 transitions)')
print(f'Transition samplee : state={t.state}, action={t.action}, reward={t.reward}')
print('TabularWorldModel mini-test OK.')


### Brique 3 : DynaQTabularAgent

Combine `QTable` + `TabularWorldModel` et ajoute :
- sélection **ε-greedy** ;
- **update réelle** : TD(0) + mémorisation dans le modèle ;
- **planning** : $k$ mises à jour simulées ;
- **bonus Dyna-Q+** : $\kappa=0$ = Dyna-Q pur.


In [ ]:
class DynaQTabularAgent:
    """Agent Dyna-Q tabulaire (kappa=0) ou Dyna-Q+ (kappa>0)."""

    def __init__(
        self, state_shape, action_count, *,
        alpha=0.1, gamma=0.99,
        epsilon=0.2, epsilon_decay=0.995, min_epsilon=0.02,
        planning_steps=10, kappa=0.0, rng=None,
    ):
        self.q_table = QTable(state_shape, action_count)
        self.action_count = int(action_count)
        self.alpha = float(alpha)
        self.gamma = float(gamma)
        self.epsilon = float(epsilon)
        self.epsilon_decay = float(epsilon_decay)
        self.min_epsilon = float(min_epsilon)
        self.planning_steps = int(planning_steps)
        self.kappa = float(kappa)
        self.rng = rng or np.random.default_rng()
        self.world_model = TabularWorldModel()
        self.total_updates = 0

    def select_action(self, state, deterministic=False):
        epsilon = 0.0 if deterministic else self.epsilon
        return epsilon_greedy(self.q_table.get(state), epsilon, self.rng)

    def td_target(self, reward, next_state, done):
        # Cible TD : r + gamma * max Q(s_prime, .)
        bootstrap = 0.0 if done else float(np.max(self.q_table.get(next_state)))
        return float(reward) + self.gamma * bootstrap

    def learn_real_transition(self, state, action, reward, next_state, done):
        target = self.td_target(reward, next_state, done)
        td_error = self.q_table.update(state, action, target, self.alpha)
        self.world_model.update(state, action, reward, next_state, done, self.total_updates)
        self.total_updates += 1
        return {'real_td_error': abs(td_error), 'planning_td_error': 0.0, 'exploration_bonus': 0.0}

    def planning_bonus(self, last_seen):
        # Dyna-Q+ : kappa * sqrt(tau), tau = pas depuis derniere visite
        tau = max(0, self.total_updates - last_seen)
        return self.kappa * math.sqrt(float(tau))

    def planning_step(self):
        transition = self.world_model.sample(self.rng)
        if transition is None:
            return {'planning_td_error': 0.0, 'exploration_bonus': 0.0}
        bonus = self.planning_bonus(transition.last_seen)
        target = self.td_target(transition.reward + bonus, transition.next_state, transition.done)
        td_error = self.q_table.update(transition.state, transition.action, target, self.alpha)
        self.total_updates += 1
        return {'planning_td_error': abs(td_error), 'exploration_bonus': float(bonus)}

    def episode_end(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)


In [ ]:
# Mini-test DynaQTabularAgent
rng_test = np.random.default_rng(42)
agent_test = DynaQTabularAgent(
    state_shape=(4, 4), action_count=2, alpha=0.2, gamma=0.9,
    epsilon=0.5, planning_steps=3, kappa=0.001, rng=rng_test,
)
m = agent_test.learn_real_transition((1, 1), 0, reward=1.0, next_state=(1, 2), done=False)
print('Apres 1 transition reelle :')
print(f'  modele size    = {len(agent_test.world_model)} (attendu 1)')
print(f'  total_updates  = {agent_test.total_updates}')
print(f'  real_td_error  = {m["real_td_error"]:.4f}')
p = agent_test.planning_step()
print(f'  planning_td_error = {p["planning_td_error"]:.4f}')
print(f'  exploration_bonus = {p["exploration_bonus"]:.6f}')
assert len(agent_test.world_model) == 1
print('DynaQTabularAgent mini-test OK.')


## 6. Pseudocode Dyna-Q et Dyna-Q+

Le squelette de Dyna-Q est très proche de Q-learning. La différence tient en une
idée : après chaque transition réelle, on met aussi à jour un **modèle du monde**,
puis on rejoue quelques transitions simulées depuis ce modèle.

On a donc deux sources d'apprentissage :

- **update réelle** : une vraie transition $(s_t, a_t, r_{t+1}, s_{t+1})$ ;
- **planning** : des transitions imaginées tirées du modèle appris $M$.

### Algorithme Dyna-Q

$$
\boxed{
\begin{aligned}
&\textbf{Initialiser } Q(s,a) \leftarrow 0 \quad \forall(s,a) \\
&\textbf{Initialiser } M \leftarrow \varnothing \\[2mm]
&\textbf{Pour chaque épisode :} \\
&\quad s \leftarrow \text{env.reset()} \\
&\quad \textbf{tant que l'épisode n'est pas terminé :} \\
&\quad\quad a \leftarrow \varepsilon\text{-greedy}(Q(s,\cdot)) \\[1mm]
&\quad\quad (r, s', done) \leftarrow \text{env.step}(a) \\[1mm]
&\quad\quad
Q(s,a) \leftarrow Q(s,a)
+ \alpha\Big[
r + \gamma \max_{a'} Q(s',a') - Q(s,a)
\Big] \\[1mm]
&\quad\quad M(s,a) \leftarrow (r, s', done) \\[2mm]
&\quad\quad \textbf{pour } i = 1,\dots,k \textbf{ :} \\
&\quad\quad\quad (\tilde{s}, \tilde{a}) \sim \text{Uniforme}(\text{clés de } M) \\
&\quad\quad\quad (\tilde{r}, \tilde{s}', \widetilde{done})
\leftarrow M(\tilde{s}, \tilde{a}) \\[1mm]
&\quad\quad\quad
Q(\tilde{s},\tilde{a}) \leftarrow Q(\tilde{s},\tilde{a})
+ \alpha\Big[
\tilde{r}
+ \gamma \mathbf{1}_{\neg \widetilde{done}}
\max_{a'} Q(\tilde{s}',a')
- Q(\tilde{s},\tilde{a})
\Big] \\[2mm]
&\quad\quad s \leftarrow s' \\[1mm]
&\quad \varepsilon \leftarrow \text{decay}(\varepsilon)
\end{aligned}
}
$$

**Lecture.** Les deux updates TD ont exactement la même forme. La seule différence
est l'origine de la transition : vraie dans la première moitié de la boucle,
simulée dans la boucle de planning. C'est pour cela que Dyna-Q est conceptuellement
simple : il ne remplace pas Q-learning, il lui ajoute une mémoire active.

Le paramètre $k$ contrôle la quantité de planning. Si $k=0$, on retombe presque sur
Q-learning. Si $k$ est grand, chaque pas réel déclenche beaucoup de mises à jour
imaginées : l'agent apprend davantage par transition réelle, mais il fait aussi
davantage confiance à son modèle.

### Variante Dyna-Q+

Dyna-Q+ garde le même squelette. Le changement se situe dans le **reward utilisé
pendant le planning**. Une transition qui n'a pas été revisitée depuis longtemps
reçoit un bonus :

$$
\tilde{r}^{+}
=
\tilde{r}
+
\kappa \sqrt{\tau(\tilde{s}, \tilde{a})}
$$

avec :

$$
\tau(\tilde{s}, \tilde{a})
=
\text{nombre de pas depuis la dernière vraie visite de }(\tilde{s}, \tilde{a})
$$

Le pas de planning devient donc :

$$
\boxed{
\begin{aligned}
&(\tilde{s}, \tilde{a}) \sim \text{Uniforme}(\text{clés de } M) \\
&(\tilde{r}, \tilde{s}', \widetilde{done}, last\_seen)
\leftarrow M(\tilde{s}, \tilde{a}) \\[1mm]
&\tau \leftarrow total\_updates - last\_seen \\[1mm]
&\tilde{r}^{+}
\leftarrow \tilde{r} + \kappa\sqrt{\tau} \\[1mm]
&Q(\tilde{s},\tilde{a}) \leftarrow Q(\tilde{s},\tilde{a})
+ \alpha\Big[
\tilde{r}^{+}
+ \gamma \mathbf{1}_{\neg \widetilde{done}}
\max_{a'} Q(\tilde{s}',a')
- Q(\tilde{s},\tilde{a})
\Big]
\end{aligned}
}
$$

**Lecture.** Dyna-Q+ introduit un optimisme contrôlé : plus une action est ancienne,
plus elle mérite d'être réessayée. Le terme $\kappa$ règle la force de cet optimisme.
Si $\kappa=0$, on retrouve Dyna-Q. Si $\kappa$ est trop grand, l'agent explore trop ;
s'il est trop petit, il risque de rester prisonnier d'un vieux modèle devenu faux.

Ce pseudocode est volontairement proche de la boucle d'entraînement qui suit :
transition réelle, update du modèle, puis $k$ updates de planning.

## 7. Entraînement tabulaire

### Boucle d'un épisode : trois phases séparées

1. **Collecte réelle** — un seul pas dans l'environnement ;
2. **Update direct** — TD(0) sur la vraie transition + mise à jour du modèle ;
3. **Planning** — $k$ mises à jour simulées.


In [ ]:
def run_tabular_episode(agent, env, discretizer, seed=None):
    obs, _ = env.reset(seed=seed)
    state = discretizer.transform(obs)
    total_reward = 0.0
    real_td_errors, planning_td_errors, bonuses = [], [], []

    for step in range(500):
        # Expérience réelle : c'est la seule interaction coûteuse avec CartPole.
        action = agent.select_action(state)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        next_state = discretizer.transform(next_obs)
        done = terminated or truncated
        total_reward += reward

        # TD réel : Q(s,a) <- Q(s,a) + alpha [r + gamma max_a' Q(s',a') - Q(s,a)].
        metrics = agent.learn_real_transition(state, action, reward, next_state, done)
        real_td_errors.append(metrics['real_td_error'])

        # Dyna-Q+ classique : les actions non essayées dans un état connu restent candidates.
        if agent.kappa > 0:
            agent.world_model.ensure_all_actions_for_state(state, agent.action_count, agent.total_updates)

        # Planning : rejouer k transitions apprises comme si elles étaient de vraies expériences.
        for _ in range(agent.planning_steps):
            plan_metrics = agent.planning_step()
            planning_td_errors.append(plan_metrics['planning_td_error'])
            bonuses.append(plan_metrics['exploration_bonus'])

        state = next_state
        if done:
            break

    agent.episode_end()
    return {
        'reward': float(total_reward),
        'length': step + 1,
        'real_td_error': float(np.mean(real_td_errors)) if real_td_errors else 0.0,
        'planning_td_error': float(np.mean(planning_td_errors)) if planning_td_errors else 0.0,
        'exploration_bonus': float(np.mean(bonuses)) if bonuses else 0.0,
    }


def evaluate_tabular_agent(agent, env_id, discretizer, episodes=8, seed=123):
    env = gym.make(env_id)
    rewards = []
    try:
        for ep in range(episodes):
            obs, _ = env.reset(seed=seed + ep)
            state = discretizer.transform(obs)
            total_reward = 0.0
            for _ in range(500):
                action = agent.select_action(state, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                state = discretizer.transform(obs)
                total_reward += reward
                if terminated or truncated:
                    break
            rewards.append(total_reward)
    finally:
        env.close()
    return float(np.mean(rewards))


def train_tabular_dyna(env_id='CartPole-v1', episodes=800, planning_steps=30, kappa=0.0, seed=7):
    env = gym.make(env_id)
    discretizer = CartPoleDiscretizer()
    agent = DynaQTabularAgent(
        state_shape=discretizer.shape,
        action_count=env.action_space.n,
        alpha=0.22, gamma=0.99,
        epsilon=0.45, epsilon_decay=0.992, min_epsilon=0.03,
        planning_steps=planning_steps, kappa=kappa,
        rng=np.random.default_rng(seed),
    )
    history = defaultdict(list)
    try:
        for episode in range(episodes):
            result = run_tabular_episode(agent, env, discretizer, seed=seed + episode)
            history['episode_reward'].append(result['reward'])
            history['episode_length'].append(result['length'])
            history['real_td_error'].append(result['real_td_error'])
            history['planning_td_error'].append(result['planning_td_error'])
            history['bonus'].append(result['exploration_bonus'])
            history['epsilon'].append(agent.epsilon)
            history['model_size'].append(len(agent.world_model))
            if (episode + 1) % 50 == 0:
                eval_r = evaluate_tabular_agent(agent, env_id, discretizer, seed=2000 + episode)
                history['eval_reward'].append(eval_r)
                history['eval_episode'].append(episode + 1)
    finally:
        env.close()
    return agent, history, discretizer

### Entraînement Dyna-Q sur CartPole

On lance un budget volontairement plus sérieux qu'un smoke test. Le tabulaire a besoin de couvrir des cases $(s,a)$ ; avec trop peu d'épisodes, on mesure surtout la pauvreté de la discrétisation, pas Dyna. Ici, chaque méthode tabulaire utilisera ensuite le **même nombre d'épisodes** pour rendre la comparaison honnête.

In [ ]:
# Configuration de la comparaison tabulaire Dyna-Q / Dyna-Q+.
SEED = 7
TABULAR_ENV_ID = "CartPole-v1"
TABULAR_EPISODES = 800
TABULAR_PLANNING_STEPS = 30

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
print(
    f"Budget tabulaire : {TABULAR_EPISODES} episodes, "
    f"{TABULAR_PLANNING_STEPS} updates de planning par pas reel"
)

dyna_q_agent, dyna_q_history, dyna_q_disc = train_tabular_dyna(
    env_id=TABULAR_ENV_ID,
    episodes=TABULAR_EPISODES,
    planning_steps=TABULAR_PLANNING_STEPS,
    kappa=0.0,
    seed=SEED,
)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
plot_learning_curve(axes[0], dyna_q_history['episode_reward'], 'Reward par episode', 'Reward', '#1f77b4', ma_window=30)
plot_learning_curve(axes[1], dyna_q_history['epsilon'], 'Decroissance de epsilon', 'epsilon', '#ff7f0e', ma_window=30)
plot_learning_curve(axes[2], dyna_q_history['real_td_error'], 'Erreur TD reelle', 'Abs delta', '#2ca02c', ma_window=30)
plot_learning_curve(axes[3], dyna_q_history['model_size'], 'Taille du modele', 'Nb transitions', '#9467bd', ma_window=30)
plt.suptitle('Diagnostics Dyna-Q sur CartPole', y=1.01)
plt.tight_layout()
plt.show()
print('Eval greedy Dyna-Q :', dyna_q_history['eval_reward'])


### Lecture des diagnostics Dyna-Q

**Reward (panneau 1).** La courbe brute est bruitée. La moyenne mobile révèle
la tendance : montée progressive vers 200+ après ~50 épisodes.

**Epsilon (panneau 2).** Décroissance exponentielle : exploration au début
(epsilon ≈ 0.35), politique quasi-greedy à la fin (epsilon ≈ 0.03).

**Erreur TD réelle (panneau 3).** Élevée au début (Q-table non calibrée), baisse
progressive. Une erreur qui reste haute indique une instabilité ou alpha trop grand.

**Taille du modèle (panneau 4).** Croît au rythme de la couverture de l'espace
$(s, a)$. Un plateau indique que la politique a convergé.


### Démo non-stationnaire : ShortcutCorridor

Dyna-Q+ est difficile à apprécier sur CartPole, car l'environnement est stationnaire : un bonus d'exploration n'a pas forcément de raison de battre Dyna-Q standard. Pour isoler l'idée, on utilise un petit corridor discret non-stationnaire.

- positions $0 \rightarrow 8$ sur une ligne ;
- départ en `0`, objectif en `8` ;
- actions : gauche, droite, **raccourci** ;
- avant l'épisode 35, le raccourci depuis la position `2` ne marche pas ;
- après l'épisode 35, ce raccourci mène directement au but.

Le piège est volontaire : un agent qui a appris que l'action « raccourci » était inutile peut cesser de l'essayer. Dyna-Q+ garde un bonus $\kappa\sqrt{\tau}$ sur les actions anciennes ; il a donc une raison de retester ce qui a vieilli.

In [ ]:
class ShortcutCorridor:
    """Corridor 1D dont un raccourci s'ouvre au milieu de l'entraînement."""

    ACTIONS = ["left", "right", "shortcut"]

    def __init__(self, switch_episode=35):
        self.length = 9
        self.start = 0
        self.goal = 8
        self.switch_episode = int(switch_episode)
        self.episode_index = 0
        self.max_steps = 30
        self.reset()

    @property
    def shortcut_open(self):
        return self.episode_index >= self.switch_episode

    def reset(self):
        self.position = self.start
        self.steps = 0
        return (self.position,)

    def step(self, action):
        if action == 0:
            self.position = max(0, self.position - 1)
        elif action == 1:
            self.position = min(self.goal, self.position + 1)
        else:
            # Avant le changement, le raccourci est un mur invisible ; après, il mène au but.
            if self.shortcut_open and self.position == 2:
                self.position = self.goal
        self.steps += 1
        done = self.position == self.goal or self.steps >= self.max_steps
        reward = 0.0 if self.position == self.goal else -1.0
        return (self.position,), reward, done


def render_shortcut_corridor(ax, env, title):
    values = np.zeros((1, env.length))
    values[0, env.goal] = 1.0
    values[0, env.start] = 0.5
    values[0, 2] = 0.25
    ax.imshow(values, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
    ax.set_title(title)
    ax.set_yticks([])
    ax.set_xticks(range(env.length))
    for x in range(env.length):
        ax.text(x, 0, str(x), ha="center", va="center", color="black")
    arrow_color = "green" if env.shortcut_open else "gray"
    ax.annotate(
        "shortcut", xy=(8, 0.18), xytext=(2, 0.18),
        arrowprops=dict(arrowstyle="->", color=arrow_color, lw=2),
        ha="center", color=arrow_color,
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 2.6))
env_pre = ShortcutCorridor(switch_episode=35)
env_post = ShortcutCorridor(switch_episode=35)
env_post.episode_index = 35
render_shortcut_corridor(axes[0], env_pre,  'Avant changement : raccourci fermé')
render_shortcut_corridor(axes[1], env_post, 'Après changement : raccourci ouvert')
plt.tight_layout()
plt.show()

### Lecture du corridor

Avant le changement, le meilleur comportement est simplement d'aller vers la droite jusqu'au but. L'action `shortcut` depuis la position `2` ne donne rien ; un agent peut donc apprendre à l'ignorer.

Après le changement, cette même action devient excellente : elle réduit brutalement le nombre de pas jusqu'au but. Dyna-Q standard ne s'en aperçoit que s'il réexplore par hasard. Dyna-Q+ pousse explicitement les actions anciennes à redevenir candidates grâce au bonus de planning.

In [ ]:
# Budget propre à l'expérience non stationnaire du corridor.
GRID_EPISODES = 70
GRID_PLANNING_STEPS = 20


In [ ]:
def train_on_changing_gridworld(episodes=GRID_EPISODES, planning_steps=GRID_PLANNING_STEPS, kappa=0.0, seed=0):
    env = ShortcutCorridor(switch_episode=35)
    rng = np.random.default_rng(seed)
    agent = DynaQTabularAgent(
        state_shape=(env.length,), action_count=3,
        alpha=0.45, gamma=0.95,
        epsilon=0.15, epsilon_decay=0.94, min_epsilon=0.0,
        planning_steps=planning_steps, kappa=kappa, rng=rng,
    )
    history = defaultdict(list)
    for episode in range(episodes):
        env.episode_index = episode
        state = env.reset()
        bonus_trace = []
        shortcut_used = 0
        for step in range(env.max_steps):
            action = agent.select_action(state)
            if state == (2,) and action == 2:
                shortcut_used += 1
            next_state, reward, done = env.step(action)
            agent.learn_real_transition(state, action, reward, next_state, done)
            if kappa > 0:
                agent.world_model.ensure_all_actions_for_state(state, agent.action_count, agent.total_updates)
            for _ in range(agent.planning_steps):
                pm = agent.planning_step()
                bonus_trace.append(pm['exploration_bonus'])
            state = next_state
            if done:
                break
        agent.episode_end()
        history['steps_to_goal'].append(step + 1)
        history['mean_bonus'].append(float(np.mean(bonus_trace)) if bonus_trace else 0.0)
        history['shortcut_tries'].append(shortcut_used)
    return agent, history


grid_dyna_agent, grid_dyna_history = train_on_changing_gridworld(
    episodes=GRID_EPISODES, planning_steps=GRID_PLANNING_STEPS, kappa=0.0, seed=SEED,
)
grid_plus_agent, grid_plus_history = train_on_changing_gridworld(
    episodes=GRID_EPISODES, planning_steps=GRID_PLANNING_STEPS, kappa=0.004, seed=SEED,
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(grid_dyna_history['steps_to_goal'], label='Dyna-Q', color='#1f77b4', alpha=0.7)
axes[0].plot(grid_plus_history['steps_to_goal'], label='Dyna-Q+', color='#d62728', alpha=0.7)
axes[0].axvline(35, linestyle='--', color='black', alpha=0.6, label='Changement')
axes[0].set_title('Pas pour atteindre le but')
axes[0].set_xlabel('Épisode')
axes[0].set_ylabel('Nombre de pas')
axes[0].legend()

axes[1].plot(grid_dyna_history['shortcut_tries'], label='Dyna-Q', color='#1f77b4')
axes[1].plot(grid_plus_history['shortcut_tries'], label='Dyna-Q+', color='#d62728')
axes[1].axvline(35, linestyle='--', color='black', alpha=0.6)
axes[1].set_title('Essais du raccourci depuis la position 2')
axes[1].set_xlabel('Épisode')
axes[1].set_ylabel('Nombre d\'essais')
axes[1].legend()

axes[2].plot(grid_plus_history['mean_bonus'], color='#9467bd')
axes[2].axvline(35, linestyle='--', color='black', alpha=0.6)
axes[2].set_title('Bonus moyen planning Dyna-Q+')
axes[2].set_xlabel('Épisode')
axes[2].set_ylabel(r'Bonus moyen $\kappa\sqrt{\tau}$')
plt.suptitle('Non-stationnarité : Dyna-Q+ réessaie les actions oubliées', y=1.01)
plt.tight_layout()
plt.show()

### Lecture détaillée de la démo non-stationnaire

La ligne verticale marque l'ouverture du raccourci. Avant ce point, l'action `shortcut` n'a aucune valeur réelle ; les deux agents apprennent surtout à avancer vers la droite. Après le changement, la question devient : **qui va retester une action devenue ancienne ?**

Le panneau central rend le mécanisme visible. Dyna-Q+ ne gagne pas par magie : son bonus augmente la valeur planifiée des actions non revisitées, donc il finit par essayer le raccourci. Une fois la nouvelle transition observée, le modèle est corrigé et le nombre de pas chute. Dyna-Q standard peut aussi découvrir le changement, mais seulement par exploration résiduelle ; si $\epsilon$ est déjà bas, il reste plus facilement prisonnier de son ancien modèle.

## 8. Diagnostics : Dyna-Q vs Dyna-Q+ sur CartPole

In [ ]:
dyna_q_plus_agent, dyna_q_plus_history, _ = train_tabular_dyna(
    env_id=TABULAR_ENV_ID,
    episodes=TABULAR_EPISODES, planning_steps=TABULAR_PLANNING_STEPS, kappa=0.0005, seed=SEED,
)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(dyna_q_history['episode_reward'], label='Dyna-Q', color='#1f77b4', alpha=0.22, linewidth=1.0)
axes[0].plot(moving_average(dyna_q_history['episode_reward'], 30), color='#1f77b4', linewidth=2.0)
axes[0].plot(dyna_q_plus_history['episode_reward'], label='Dyna-Q+', color='#d62728', alpha=0.22, linewidth=1.0)
axes[0].plot(moving_average(dyna_q_plus_history['episode_reward'], 30), color='#d62728', linewidth=2.0)
axes[0].set_title('Reward par episode')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward')
axes[0].legend()

axes[1].plot(dyna_q_history['eval_episode'], dyna_q_history['eval_reward'], 'o-', label='Dyna-Q', color='#1f77b4')
axes[1].plot(dyna_q_plus_history['eval_episode'], dyna_q_plus_history['eval_reward'], 's-', label='Dyna-Q+', color='#d62728')
axes[1].set_title('Evaluation greedy periodique')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Reward moyen')
axes[1].legend()

axes[2].plot(dyna_q_plus_history['bonus'], color='#8c564b')
axes[2].set_title('Bonus moyen Dyna-Q+')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel(r'Bonus moyen $\kappa\sqrt{\tau}$')
plt.suptitle('Dyna-Q vs Dyna-Q+ sur CartPole (meme budget, meme seed)', y=1.01)
plt.tight_layout()
plt.show()
print('Eval greedy Dyna-Q   :', dyna_q_history['eval_reward'])
print('Eval greedy Dyna-Q+  :', dyna_q_plus_history['eval_reward'])


### Lecture des diagnostics comparatifs

**Reward train.** Les courbes restent bruitées car CartPole démarre dans des états différents et l'agent explore encore. On lit donc surtout la moyenne mobile.

**Évaluation greedy.** C'est le panneau central qui compte le plus : même politique déterministe, mêmes seeds d'évaluation, pas de bruit d'exploration. Sur un environnement stationnaire comme CartPole, Dyna-Q+ n'a pas vocation à écraser Dyna-Q ; il peut être similaire, parfois un peu meilleur ou un peu pire selon le bonus.

**Bonus.** Le bonus Dyna-Q+ reste faible ici parce que les transitions utiles sont souvent revisitées. C'est cohérent avec la démo du corridor : le bonus est surtout un outil pour les environnements où le vieux modèle peut devenir faux ou incomplet.

## 9. Deep Dyna bloc par bloc

### Pourquoi passer au deep learning ?

1. **L'espace d'état doit être discret** pour le cas tabulaire ;
2. **Pas de généralisation** : deux états proches ont des Q-values indépendantes.

| Composant tabulaire    | Composant deep                     |
|------------------------|------------------------------------|
| `QTable`               | `QNetwork` (MLP)                   |
| `TabularWorldModel`    | `DynamicsNetwork` (MLP)            |
| Replay direct          | `ReplayBuffer` + batch sampling    |
| Update TD              | Perte Bellman (DQN)                |


### Brique deep 1 : QNetwork

Avec un réseau, $Q_\theta(s, \cdot)$ prend $s \in \mathbb{R}^4$ et produit
$n_\text{actions}$ valeurs. Cette généralisation permet de gérer des espaces
continus — au prix d'une instabilité potentielle (d'où le réseau cible $Q_{\bar\theta}$).


In [ ]:
class QNetwork(nn.Module):
    """MLP obs -> Q(s, .) pour toutes les actions en un seul passage."""

    def __init__(self, obs_dim, action_count, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_count),
        )

    def forward(self, obs):
        # obs : (batch, obs_dim)  ->  sortie : (batch, action_count)
        return self.net(obs)


In [ ]:
# Mini-test QNetwork
q_net = QNetwork(obs_dim=4, action_count=2, hidden_dim=32)
obs_batch = torch.randn(8, 4)
q_values = q_net(obs_batch)
print('Entree      :', obs_batch.shape)
print('Sortie Q    :', q_values.shape)
assert q_values.shape == (8, 2)
best_actions = q_values.argmax(dim=1)
print('Actions     :', best_actions.tolist())
print('QNetwork mini-test OK.')


### Brique deep 2 : DynamicsNetwork

Le réseau de dynamiques apprend à prédire $(r, s', \text{done})$ depuis $(s, a)$ :

$$M_\phi : (s, a) \;\to\; (\hat{r},\; \Delta s,\; \hat{d}_{\text{logit}})$$

Choix de design :
1. **Encodage action en one-hot** : $n_\text{actions}$ dimensions binaires.
2. **Prédiction résiduelle** $\hat{s}' = s + \Delta$ : apprend la *différence*.


In [ ]:
class DynamicsNetwork(nn.Module):
    """MLP (s, a_one_hot) -> (next_obs, reward, done_logit). Prediction residuelle."""

    def __init__(self, obs_dim, action_count, hidden_dim=64):
        super().__init__()
        self.action_count = int(action_count)
        input_dim = obs_dim + action_count
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.delta_head  = nn.Linear(hidden_dim, obs_dim)
        self.reward_head = nn.Linear(hidden_dim, 1)
        self.done_head   = nn.Linear(hidden_dim, 1)

    def forward(self, obs, actions):
        one_hot = F.one_hot(actions.long(), num_classes=self.action_count).float()
        features = self.encoder(torch.cat([obs, one_hot], dim=1))
        # Prediction residuelle : le reseau apprend Delta_s, pas s_prime directement.
        delta      = self.delta_head(features)
        next_obs   = obs + delta
        reward     = self.reward_head(features).squeeze(1)
        done_logit = self.done_head(features).squeeze(1)
        return next_obs, reward, done_logit


In [ ]:
# Mini-test DynamicsNetwork
dyn_net = DynamicsNetwork(obs_dim=4, action_count=2, hidden_dim=32)
obs_b  = torch.randn(6, 4)
acts_b = torch.randint(0, 2, (6,))
next_obs_b, reward_b, done_b = dyn_net(obs_b, acts_b)
print("Entree obs   :", obs_b.shape)
print("Sortie s'    :", next_obs_b.shape)
print("Sortie r     :", reward_b.shape)
print("Sortie done  :", done_b.shape)
assert next_obs_b.shape == (6, 4)
assert reward_b.shape == done_b.shape == (6,)
print('Delta moyen  :', (next_obs_b - obs_b).abs().mean().item())
print('DynamicsNetwork mini-test OK.')


### Brique deep 3 : ReplayBuffer

Stocke les vraies transitions pour :
1. L'**update DQN** (perte Bellman sur batch réel) ;
2. L'**entraînement du modèle** ($\mathcal{L}_M$ sur batch réel) ;
3. Le **planning** (générer des transitions imaginées depuis des états réels).


In [ ]:
class ReplayBuffer:
    """Buffer FIFO circulaire. Stocke des (s, a, r, s_prime, done) NumPy."""

    def __init__(self, capacity, rng=None):
        self.capacity = int(capacity)
        self.rng = rng or np.random.default_rng()
        self.storage = deque(maxlen=self.capacity)

    def push(self, state, action, reward, next_state, done):
        self.storage.append((
            np.asarray(state,      dtype=np.float32),
            int(action), float(reward),
            np.asarray(next_state, dtype=np.float32),
            float(done),
        ))

    def sample(self, batch_size):
        idx = self.rng.choice(len(self.storage), size=batch_size, replace=False)
        batch = [self.storage[i] for i in idx]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(np.asarray(states),      dtype=torch.float32),
            torch.tensor(np.asarray(actions),     dtype=torch.long),
            torch.tensor(np.asarray(rewards),     dtype=torch.float32),
            torch.tensor(np.asarray(next_states), dtype=torch.float32),
            torch.tensor(np.asarray(dones),       dtype=torch.float32),
        )

    def __len__(self):
        return len(self.storage)


In [ ]:
# Mini-test ReplayBuffer
buf = ReplayBuffer(capacity=20, rng=np.random.default_rng(0))
for i in range(15):
    obs  = np.random.randn(4).astype(np.float32)
    nobs = np.random.randn(4).astype(np.float32)
    buf.push(obs, i % 2, float(i), nobs, i == 14)
print(f'Buffer size : {len(buf)} (attendu 15)')
states, actions, rewards, next_states, dones = buf.sample(8)
print(f'states shape  : {states.shape}')
print(f'actions shape : {actions.shape}')
print(f'rewards range : [{rewards.min():.1f}, {rewards.max():.1f}]')
assert states.shape == (8, 4) and actions.dtype == torch.long
print('ReplayBuffer mini-test OK.')


### Brique deep 4 : DeepDynaAgent

À chaque pas d'entraînement :

1. **Stocker** la transition réelle dans le buffer ;
2. **Update DQN** sur un mini-batch réel (perte Bellman) ;
3. **Entraîner le modèle** (MSE état + MSE reward + BCE done) ;
4. **Planning imaginé** : $k$ transitions via le modèle → update DQN.


In [ ]:
class DeepDynaAgent:
    """Partie structurelle : reseaux, replay, epsilon-greedy et stockage des transitions."""

    def __init__(
        self, obs_dim, action_count, *,
        hidden_dim=64, gamma=0.99,
        lr=1e-3, model_lr=1e-3,
        epsilon=1.0, epsilon_decay=0.97, min_epsilon=0.05,
        batch_size=32, buffer_capacity=5000,
        start_learning_after=64,
        imagined_updates=3, model_train_steps=1,
        target_update_freq=50, seed=0,
    ):
        self.gamma = float(gamma)
        self.epsilon = float(epsilon)
        self.epsilon_decay = float(epsilon_decay)
        self.min_epsilon = float(min_epsilon)
        self.batch_size = int(batch_size)
        self.start_learning_after = int(start_learning_after)
        self.imagined_updates = int(imagined_updates)
        self.model_train_steps = int(model_train_steps)
        self.target_update_freq = int(target_update_freq)
        self.rng = np.random.default_rng(seed)
        self.update_steps = 0
        self.q_online  = QNetwork(obs_dim, action_count, hidden_dim)
        self.q_target  = QNetwork(obs_dim, action_count, hidden_dim)
        self.q_target.load_state_dict(self.q_online.state_dict())
        self.q_target.eval()
        self.dynamics  = DynamicsNetwork(obs_dim, action_count, hidden_dim)
        self.q_optim     = torch.optim.Adam(self.q_online.parameters(), lr=lr)
        self.model_optim = torch.optim.Adam(self.dynamics.parameters(), lr=model_lr)
        self.replay = ReplayBuffer(buffer_capacity, rng=self.rng)

    def select_action(self, obs, deterministic=False):
        if (not deterministic) and self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.q_online.net[-1].out_features))
        obs_t = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            q = self.q_online(obs_t)
        return int(q.argmax(dim=1).item())

    def store(self, state, action, reward, next_state, done):
        self.replay.push(state, action, reward, next_state, done)

    def sample_batch(self):
        return self.replay.sample(self.batch_size)

    def episode_end(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)

    
    """Ajoute les trois updates explicites : Q reelle, modele et planning imagine."""

    def _q_update(self, batch):
        """Perte Bellman sur un batch de transitions."""
        states, actions, rewards, next_states, dones = batch
        q_values = self.q_online(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q   = self.q_target(next_states).max(dim=1).values
            # Cible Bellman : r + gamma * max_a_prime Q_bar(s_prime, a_prime) * (1-done)
            targets  = rewards + self.gamma * (1.0 - dones) * next_q
        td_errors = targets - q_values
        loss = F.mse_loss(q_values, targets)
        self.q_optim.zero_grad()
        loss.backward()
        self.q_optim.step()
        self.update_steps += 1
        if self.update_steps % self.target_update_freq == 0:
            self.q_target.load_state_dict(self.q_online.state_dict())
        return float(loss.item()), float(td_errors.abs().mean().item())

    def _model_update(self, batch):
        """Entraine le modele : MSE etat + MSE reward + BCE done."""
        states, actions, rewards, next_states, dones = batch
        pred_next, pred_reward, pred_done_logit = self.dynamics(states, actions)
        state_loss  = F.mse_loss(pred_next, next_states)
        reward_loss = F.mse_loss(pred_reward, rewards)
        done_loss   = F.binary_cross_entropy_with_logits(pred_done_logit, dones)
        total_loss  = state_loss + reward_loss + done_loss
        self.model_optim.zero_grad()
        total_loss.backward()
        self.model_optim.step()
        return float(state_loss.item()), float(reward_loss.item()), float(done_loss.item())

    def _imagined_q_update(self, batch):
        """Genere une transition imaginee via le modele, puis fait un update Q."""
        states, _, _, _, _ = batch
        with torch.no_grad():
            imagined_actions   = self.q_online(states).argmax(dim=1)
            pred_next, pred_r, pred_done_logit = self.dynamics(states, imagined_actions)
            pred_done = (torch.sigmoid(pred_done_logit) > 0.5).float()
        _, td = self._q_update((states, imagined_actions, pred_r, pred_next, pred_done))
        return td

    def learn_step(self):
        """Un pas complet : Q reel + modele + k updates imaginees."""
        ready = max(self.batch_size, self.start_learning_after)
        if len(self.replay) < ready:
            return {'q_loss': 0.0, 'real_td_error': 0.0,
                    'model_state_loss': 0.0, 'model_reward_loss': 0.0,
                    'model_done_loss': 0.0, 'planning_td_error': 0.0}
        q_loss, real_td = self._q_update(self.sample_batch())
        s_losses, r_losses, d_losses = [], [], []
        for _ in range(self.model_train_steps):
            sl, rl, dl = self._model_update(self.sample_batch())
            s_losses.append(sl); r_losses.append(rl); d_losses.append(dl)
        imag_tds = [self._imagined_q_update(self.sample_batch())
                    for _ in range(self.imagined_updates)]
        return {
            'q_loss':            q_loss,
            'real_td_error':     real_td,
            'model_state_loss':  float(np.mean(s_losses)),
            'model_reward_loss': float(np.mean(r_losses)),
            'model_done_loss':   float(np.mean(d_losses)),
            'planning_td_error': float(np.mean(imag_tds)) if imag_tds else 0.0,
        }



In [ ]:
# Mini-test DeepDynaAgent
deep_test = DeepDynaAgent(obs_dim=4, action_count=2, hidden_dim=32,
                          batch_size=8, buffer_capacity=500,
                          start_learning_after=8, seed=0)
rng_t = np.random.default_rng(0)
for _ in range(20):
    o  = rng_t.standard_normal(4).astype(np.float32)
    a  = int(rng_t.integers(2))
    no = rng_t.standard_normal(4).astype(np.float32)
    deep_test.store(o, a, 1.0, no, False)
metrics = deep_test.learn_step()
print('Metriques d un pas d apprentissage :')
for k, v in metrics.items():
    print(f'  {k:<22} = {v:.6f}')
assert 'q_loss' in metrics and 'model_state_loss' in metrics
print('DeepDynaAgent mini-test OK.')


### Interprétation : planning deep vs planning tabulaire

Dans Dyna tabulaire, le planning rejoue des transitions deja memorisees :

$$
M(s,a) = (r, s', done)
$$

En Deep Dyna, le modele n'est plus un dictionnaire. C'est un reseau de dynamique
$M_\phi$ qui predit une transition a partir d'un etat continu et d'une action :

$$
M_\phi(s,a)
=
(\hat{r}, \hat{s}', \hat{d})
$$

Le planning ne consiste donc plus seulement a rejouer une vieille transition. Il
peut aussi produire une transition plausible dans une region proche des donnees.

### Pas de planning Deep Dyna

$$
\boxed{
\begin{aligned}
&\textbf{Echantillonner un etat reel depuis le replay buffer :} \\
&\quad s \sim \mathcal{D} \\[1mm]
&\textbf{Choisir l'action actuellement preferee par le Q-network :} \\
&\quad a^\star \leftarrow \arg\max_a Q_\theta(s,a) \\[1mm]
&\textbf{Interroger le modele de dynamique appris :} \\
&\quad (\hat{r}, \hat{s}', \hat{d}) \leftarrow M_\phi(s,a^\star) \\[1mm]
&\textbf{Construire la cible Bellman imaginee :} \\
&\quad y_{\text{imag}} \leftarrow \hat{r} + \gamma(1-\hat{d}) \max_{a'} Q_{\bar{\theta}}(\hat{s}',a') \\[1mm]
&\textbf{Mettre a jour le Q-network :} \\
&\quad \theta \leftarrow \theta - \eta \nabla_\theta \Big(Q_\theta(s,a^\star) - y_{\text{imag}}\Big)^2
\end{aligned}
}
$$

**Lecture.** La structure de controle reste la meme que dans Dyna tabulaire :
fabriquer une transition, puis appliquer une update Bellman. Ce qui change, c'est la
source de cette transition. En tabulaire, elle est relue ; en deep, elle est predite.

**Progression pedagogique.** On ne passe donc pas d'un coup de Dyna-Q a un autre
algorithme. On remplace brique par brique :

1. `Q-table` -> `QNetwork` pour generaliser en etat continu ;
2. dictionnaire de transitions -> modele neuronal ;
3. planning par relecture -> planning par prediction.

**Risque central.** Cette progression gagne en generalisation, mais elle ouvre une
nouvelle voie d'erreur : si le modele hallucine une transition plausible mais fausse,
la mise a jour de Q devient elle aussi fausse. Deep Dyna doit donc surveiller
separement la perte Bellman reelle, les pertes du modele, et l'erreur TD imaginee.

## Entrainement Deep Dyna sur CartPole

In [ ]:
# Configuration de l'extension Deep Dyna.
DEEP_ENV_ID = "CartPole-v1"
DEEP_DYNA_EPISODES = 800
DEEP_DYNA_EVAL_EVERY = 15


In [ ]:
def evaluate_deep_agent(agent, env_id=DEEP_ENV_ID, episodes=5, seed=100, show_progress=False):
    env = gym.make(env_id)
    rewards = []
    iterator = range(episodes)
    if show_progress:
        iterator = tqdm(iterator, desc="Eval Deep Dyna", leave=False)

    try:
        for ep in iterator:
            obs, _ = env.reset(seed=seed + ep)
            total_reward = 0.0

            for _ in range(500):
                action = agent.select_action(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                if terminated or truncated:
                    break

            rewards.append(total_reward)

            if show_progress:
                iterator.set_postfix(mean=f"{np.mean(rewards):.1f}", last=f"{total_reward:.1f}")

    finally:
        env.close()

    return float(np.mean(rewards))


def run_deep_dyna_episode(agent, env, seed):
    obs, _ = env.reset(seed=seed)
    total_reward = 0.0
    ep_metrics = []

    for _ in range(500):
        action = agent.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.store(obs, action, reward, next_obs, done)
        metrics = agent.learn_step()
        ep_metrics.append(metrics)

        obs = next_obs
        total_reward += reward

        if done:
            break

    agent.episode_end()
    return float(total_reward), ep_metrics


def train_deep_dyna_cartpole(episodes=60, seed=7):
    env = gym.make(DEEP_ENV_ID)
    agent = DeepDynaAgent(
        obs_dim=env.observation_space.shape[0],
        action_count=env.action_space.n,
        hidden_dim=64,
        gamma=0.99,
        lr=1e-3,
        model_lr=1e-3,
        epsilon=1.0,
        epsilon_decay=0.96,
        min_epsilon=0.05,
        batch_size=32,
        buffer_capacity=4000,
        start_learning_after=64,
        imagined_updates=3,
        model_train_steps=1,
        target_update_freq=60,
        seed=seed,
    )

    history = defaultdict(list)
    pbar = tqdm(range(episodes), desc="Training Deep Dyna", unit="episode")

    try:
        for episode in pbar:
            total_reward, ep_metrics = run_deep_dyna_episode(
                agent,
                env,
                seed=seed + episode,
            )

            history["episode_reward"].append(total_reward)
            history["epsilon"].append(agent.epsilon)

            if ep_metrics:
                for key in ep_metrics[-1].keys():
                    values = [x[key] for x in ep_metrics if key in x and np.isfinite(x[key])]
                    if values:
                        history[key].append(float(np.mean(values)))

            postfix = {
                "reward": f"{total_reward:.1f}",
                "avg10": f"{np.mean(history['episode_reward'][-10:]):.1f}",
                "eps": f"{agent.epsilon:.3f}",
            }

            if (episode + 1) % DEEP_DYNA_EVAL_EVERY == 0:
                eval_reward = evaluate_deep_agent(agent)
                history["eval_reward"].append(eval_reward)
                history["eval_episode"].append(episode + 1)

                print(
                    f"[Deep Dyna] episode {episode + 1:>3}/{episodes} | "
                    f"train_reward={total_reward:>6.1f} | "
                    f"avg10={np.mean(history['episode_reward'][-10:]):>6.1f} | "
                    f"eval={eval_reward:>6.1f} | "
                    f"epsilon={agent.epsilon:.3f}"
                )

                postfix["eval"] = f"{eval_reward:.1f}"

            pbar.set_postfix(postfix)

    finally:
        env.close()

    return agent, history

In [ ]:

deep_dyna_agent, deep_history = train_deep_dyna_cartpole(episodes=DEEP_DYNA_EPISODES, seed=SEED)
print('Eval greedy Deep Dyna :', deep_history['eval_reward'])

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
plot_learning_curve(axes[0, 0], deep_history['episode_reward'],
                    'Reward par episode', 'Reward', '#1f77b4')
plot_learning_curve(axes[0, 1], deep_history['q_loss'],
                    'Perte Bellman (Q reel)', 'MSE', '#ff7f0e')
plot_learning_curve(axes[0, 2], deep_history['model_state_loss'],
                    'Perte modele next-state (MSE)', 'MSE', '#2ca02c')
plot_learning_curve(axes[1, 0], deep_history['model_reward_loss'],
                    'Perte modele reward (MSE)', 'MSE', '#d62728')
plot_learning_curve(axes[1, 1], deep_history['model_done_loss'],
                    'Perte modele done (BCE)', 'BCE', '#9467bd')
plot_learning_curve(axes[1, 2], deep_history['planning_td_error'],
                    'Erreur TD planning (imagine)', 'Abs delta', '#8c564b')
plt.suptitle('Diagnostics Deep Dyna -- CartPole', y=1.01)
plt.tight_layout()
plt.show()


### Lecture panneau par panneau des diagnostics Deep Dyna

**Reward (haut gauche).** La progression est plus irrégulière qu'en tabulaire :
le réseau Q et le modèle se co-entraînent. Montée attendue après ~30 épisodes.
Un effondrement soudain indique des transitions imaginées trop biaisées.

**Perte Bellman Q réel (haut milieu).** Doit diminuer progressivement. Une perte
qui reste haute signale que le réseau ne converge pas.

**Perte next-state (haut droit).** Les deltas sont petits pour CartPole → perte
doit atteindre des valeurs très faibles (<0.01). Si elle reste haute, les transitions
imaginées seront biaisées.

**Perte reward (bas gauche).** CartPole donne un reward constant de $+1$ sauf à la
fin — le modèle doit apprendre un champ quasi-constant, converge rapidement.

**Perte done (bas milieu).** La classe "done=True" est très rare. Un modèle qui
prédit mal les fins d'épisode génèrera des transitions imaginées sans fin.

**Erreur TD planning (bas droit).** Si le modèle converge bien, doit être similaire
à l'erreur TD réelle. Un écart important indique un modèle insuffisant.


## 10. Comparaison des variantes

La promesse du RL model-based est d'exploiter davantage chaque transition reelle. On
compare maintenant les trois variantes principales du chapitre -- Dyna-Q, Dyna-Q+ et
Deep Dyna -- avec une baseline Q-learning pure. Pour etre honnete, les methodes
tabulaires utilisent le **meme nombre d'episodes** ; seule la quantite de planning
change. Deep Dyna prolonge la meme idee de planning, mais avec une representation
continue et un cout d'optimisation different : il faut donc comparer la vitesse
d'apprentissage et les diagnostics, pas seulement la meilleure courbe finale.

In [ ]:
q_only_agent, q_only_history, _ = train_tabular_dyna(
    env_id=TABULAR_ENV_ID,
    episodes=TABULAR_EPISODES, planning_steps=0, kappa=0.0, seed=SEED,
)
print('Eval greedy Q-learning  :', q_only_history['eval_reward'])
print('Eval greedy Dyna-Q      :', dyna_q_history['eval_reward'])
print('Eval greedy Dyna-Q+     :', dyna_q_plus_history['eval_reward'])
print('Eval greedy Deep Dyna   :', deep_history['eval_reward'])


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors = {'Q-learning': '#555555', 'Dyna-Q': '#1f77b4', 'Dyna-Q+': '#d62728', 'Deep Dyna': '#2ca02c'}
histories = {
    'Q-learning': q_only_history['episode_reward'],
    'Dyna-Q': dyna_q_history['episode_reward'],
    'Dyna-Q+': dyna_q_plus_history['episode_reward'],
    'Deep Dyna': deep_history['episode_reward'],
}
for label, vals in histories.items():
    c = colors[label]
    window = 30 if label != 'Deep Dyna' else 10
    axes[0].plot(vals, color=c, alpha=0.20, linewidth=1.0)
    axes[0].plot(moving_average(vals, window), color=c, linewidth=2.2, label=label)
axes[0].set_title('Reward train -- comparaison')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward')
axes[0].legend()

for label, history in [
    ('Q-learning', q_only_history),
    ('Dyna-Q', dyna_q_history),
    ('Dyna-Q+', dyna_q_plus_history),
    ('Deep Dyna', deep_history),
]:
    axes[1].plot(history['eval_episode'], history['eval_reward'], 'o-', color=colors[label], label=label)
axes[1].set_title('Evaluation greedy periodique')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Reward moyen')
axes[1].legend(fontsize=8)

methods = ['Q-learning', 'Dyna-Q', 'Dyna-Q+', 'Deep Dyna']
repr_types = ['Table discrete', 'Table discrete', 'Table discrete', 'MLP continu']
planning = ['k=0', f'k={TABULAR_PLANNING_STEPS}', f'k={TABULAR_PLANNING_STEPS}, bonus', 'transitions imaginees']
best_evals = [
    f"{max(q_only_history['eval_reward']):.0f}" if q_only_history['eval_reward'] else 'N/A',
    f"{max(dyna_q_history['eval_reward']):.0f}" if dyna_q_history['eval_reward'] else 'N/A',
    f"{max(dyna_q_plus_history['eval_reward']):.0f}" if dyna_q_plus_history['eval_reward'] else 'N/A',
    f"{max(deep_history['eval_reward']):.0f}" if deep_history['eval_reward'] else 'N/A',
]
sample_reuse = ['1 update / transition', '1 + planning tabulaire', '1 + planning + bonus', '1 + planning neuronal']
axes[2].axis('off')
table = axes[2].table(
    cellText=list(zip(methods, repr_types, planning, sample_reuse, best_evals)),
    colLabels=['Methode', 'Representation', 'Planning', 'Reutilisation', 'Meilleure eval'],
    cellLoc='center', loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8.0)
table.scale(1.3, 1.5)
axes[2].set_title('Tableau recapitulatif', pad=12)
plt.suptitle('Dyna-Q, Dyna-Q+ et Deep Dyna : meme question, trois reponses', y=1.01)
plt.tight_layout()
plt.show()

### Lecture du tableau comparatif

**Q-learning pur.** C'est la baseline sans modele. Une transition reelle produit une
seule mise a jour, donc la courbe sert de reference minimale sur l'utilite de la
discretisation.

**Dyna-Q.** Le gain attendu n'est pas de trouver une autre politique, mais de monter
plus vite grace au planning. La comparaison utile est donc a budget d'episodes egal.

**Dyna-Q+.** Sur `CartPole-v1`, son bonus peut sembler discret. C'est normal : son
avantage apparait surtout quand le monde change ou quand certaines actions meritent
d'etre re-testees apres une longue absence.

**Deep Dyna.** Il garde l'idee Dyna mais change la representation et le modele.
L'important n'est pas de le sacrer "meilleur" a tout prix ; c'est de voir ce qu'on
gagne (generalisation) et ce qu'on paie (variance, erreurs de modele, cout
d'optimisation supplementaire).

## 11. Demo finale

In [ ]:
def evaluate_and_display_agent(agent, label, discretizer=None, n_episodes=3, seed=42, video_root="videos/09_dyna_cartpole"):
    """Évalue une politique greedy CartPole et affiche le dernier replay vidéo enregistré."""
    safe_label = label.lower().replace(" ", "_").replace("-", "_").replace("+", "plus")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    env_eval = gym.make("CartPole-v1", render_mode="rgb_array")
    env_eval = RecordVideo(
        env_eval,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == n_episodes - 1,
        name_prefix=f"{safe_label}_greedy",
    )

    rewards = []
    last_trace = []

    try:
        for episode in range(n_episodes):
            obs, _ = env_eval.reset(seed=seed + episode)
            total_reward = 0.0
            trace = []

            for step in range(500):
                if discretizer is None:
                    action = agent.select_action(obs, deterministic=True)
                else:
                    state = discretizer.transform(obs)
                    action = agent.select_action(state, deterministic=True)

                next_obs, reward, terminated, truncated, _ = env_eval.step(action)
                total_reward += reward
                trace.append({"step": step, "x": obs[0], "theta": obs[2], "action": action})
                obs = next_obs

                if terminated or truncated:
                    break

            rewards.append(total_reward)
            last_trace = trace
            print(f"[{label}] Épisode {episode + 1} : reward={total_reward:.0f} / 500")

    finally:
        env_eval.close()

    print(f"[{label}] Moyenne : {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}")

    if last_trace:
        steps_t = [item["step"] for item in last_trace]
        theta_t = [item["theta"] * 180 / math.pi for item in last_trace]
        x_t = [item["x"] for item in last_trace]
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(steps_t, theta_t, color="#1f77b4")
        axes[0].axhline(12, linestyle="--", color="red", alpha=0.5, label="Limite +/-12 deg")
        axes[0].axhline(-12, linestyle="--", color="red", alpha=0.5)
        axes[0].set_title("Angle du pendule theta (deg)")
        axes[0].set_xlabel("Pas")
        axes[0].set_ylabel("theta (degrés)")
        axes[0].legend()
        axes[1].plot(steps_t, x_t, color="#ff7f0e")
        axes[1].axhline(2.4, linestyle="--", color="red", alpha=0.5, label="Limite +/-2.4")
        axes[1].axhline(-2.4, linestyle="--", color="red", alpha=0.5)
        axes[1].set_title("Position du chariot x")
        axes[1].set_xlabel("Pas")
        axes[1].set_ylabel("x")
        axes[1].legend()
        plt.suptitle(f"Trajectoire greedy -- {label} (reward = {rewards[-1]:.0f})", y=1.01)
        plt.tight_layout()
        plt.show()

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return rewards


demo_results = {}
if dyna_q_agent is not None:
    demo_results["Dyna-Q"] = evaluate_and_display_agent(dyna_q_agent, "Dyna-Q", discretizer=dyna_q_disc)
if dyna_q_plus_agent is not None:
    demo_results["Dyna-Q+"] = evaluate_and_display_agent(dyna_q_plus_agent, "Dyna-Q+", discretizer=dyna_q_disc)
if deep_dyna_agent is not None:
    demo_results["Deep Dyna"] = evaluate_and_display_agent(deep_dyna_agent, "Deep Dyna", discretizer=None)

print("Résumé démo greedy :")
for name, values in demo_results.items():
    print(f"  {name:10s} : {np.mean(values):.1f} +/- {np.std(values):.1f}")


### Lecture de la trajectoire greedy

**Angle theta (panneau gauche).** Un agent bien entraîné maintient theta entre ±5°.
De légères oscillations sont normales — CartPole est sous-amorti. Si theta dépasse
±10° régulièrement, l'agent est encore instable.

**Position x (panneau droit).** Le chariot doit rester proche du centre. Une dérive
vers les bords indique une politique insuffisamment corrective — limite connue de
la discrétisation grossière.

Si le reward final est 500, l'agent a tenu tout l'épisode.


## References

- Sutton, R. S. (1990). Integrated architectures for learning, planning, and reacting based on approximating dynamic programming. *ICML 1990*.
- Sutton, R. S. & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.), chapitre 8.
- Ce segment Deep Dyna est une continuation pedagogique du schema Dyna : il montre comment remplacer table et modele tabulaire par des reseaux, sans pretendre reproduire un papier canonique unique a l'identique.

## Limites et perspectives

### Ce que Dyna fait bien

- **Efficacité-échantillon** : apprendre plus vite avec moins d'interactions réelles.
- **Extensibilité** : même tronc dans n'importe quel environnement discret.
- **Transparence** : composants séparés et diagnosticables indépendamment.

### Limites

| Limite                              | Implication                                           |
|-------------------------------------|-------------------------------------------------------|
| **Discrétisation grossière** (tab.) | Perte d'information, mauvaise généralisation spatiale |
| **Modèle déterministe à 1 pas**     | Ne gère pas le bruit, les dynamiques multimodales     |
| **Pas de généralisation** (tab.)    | Chaque $(s,a)$ est appris indépendamment              |
| **Propagation d'erreur** (deep)     | Les erreurs du modèle se propagent dans Q             |
| **Monde stationnaire implicite**    | Le modèle n'oublie pas sauf avec Dyna-Q+              |

### Vers les méthodes avancées

- **PILCO** (2011) — modèle probabiliste (processus gaussien) ;
- **PETS** — ensemble de réseaux pour l'incertitude aléatoire ;
- **MBPO** (2019) — modèle deep + policy gradient ;
- **Dreamer** — représentations latentes récurrentes.

Dans tous ces cas, le principe Dyna reste le même : apprendre un modèle, planifier
dessus, corriger avec des données réelles.


## Récapitulatif

| Idée clé                      | Ce qu'on a vu                                                  |
|-------------------------------|----------------------------------------------------------------|
| **Modèle du monde**           | Dictionnaire $(s,a) \to (r, s')$ ou réseau $M_\phi$           |
| **Planning**                  | $k$ updates TD sur transitions simulées après chaque pas réel |
| **Même update**               | La mise à jour TD est identique pour réel et simulé            |
| **Bonus $\kappa\sqrt{\tau}$** | Dyna-Q+ : favorise les paires $(s,a)$ vieilles                |
| **Prédiction résiduelle**     | Deep Dyna apprend $\Delta s$ plutôt que $s'$ directement      |
| **Trois pertes**              | MSE état + MSE reward + BCE done pour $M_\phi$                |
| **Efficacité-échantillon**    | $k > 0$ améliore l'apprentissage avec moins d'interactions    |
